<div style="border:solid purple 2px; padding: 20px">

Привет Сергей! 👋

Меня зовут Рустам Муртазин, и я буду делать ревью твоей работы. Давай будем общаться на **«ты»**. Если это неприемлемо, обязательно напиши мне в комментариях — мы перейдем на **«вы»**.

Я не хочу указывать тебе на совершенные тобою ошибки, а хочу поделиться своим опытом и помочь тебе стать настоящим профессионалом и сделать проект еще лучше.

Обрати внимание в первую очередь на те, что помечены <span style="color:red">красным цветом</span>. После их доработки проект будет принят. <span style="color:green">Зеленым цветом</span> отмечены удачные и элегантные решения, на которые можно опираться в будущих проектах. <span style="color:orange">Оранжевым цветом</span> выделено то, что в следующий раз можно сделать по-другому. Ты можешь учесть эти комментарии при выполнении будущих заданий или доработать проект сейчас (однако это не обязательно). Также в проекте могут быть небольшие «лайфхаки» по Python, не относящиеся к проекту, их я выделил фиолетовым цветом)

Давай работать над проектом в диалоге: если ты **что-то меняешь** в проекте по моим рекомендациям — **пиши об этом**. Выбери для своих комментариев какой-то заметный цвет, так мне будет легче отследить изменения. Пожалуйста, **не перемещай, не изменяй и не удаляй мои комментарии**. Всё это поможет выполнить повторную проверку твоего проекта оперативнее».

---

Сергей, классная работа! Код чистый, структурированный, понятный. По всей работе прослеживается логика принятий решений. В работе нет критических замечаний и я ее принимаю (так как проверка и так затянулась...). При этом, если у тебя остануться какие-то вопросы, то ты можешь задать их мне через куратора) В общем получилась классная работа. Благодарю за старания и желаю успехов в дальнейших проектах 😊

# Выбор локации для скважины

Допустим, вы работаете в добывающей компании «ГлавРосГосНефть». Нужно решить, где бурить новую скважину.

Вам предоставлены пробы нефти в трёх регионах: в каждом 100 000 месторождений, где измерили качество нефти и объём её запасов. Постройте модель машинного обучения, которая поможет определить регион, где добыча принесёт наибольшую прибыль. Проанализируйте возможную прибыль и риски техникой *Bootstrap.*

Шаги для выбора локации:

- В избранном регионе ищут месторождения, для каждого определяют значения признаков;
- Строят модель и оценивают объём запасов;
- Выбирают месторождения с самым высокими оценками значений. Количество месторождений зависит от бюджета компании и стоимости разработки одной скважины;
- Прибыль равна суммарной прибыли отобранных месторождений.

## Служебные файлы

### Загружаем все модули Python

In [1]:
#Загружаем все модули
#!pip install phik -q
#!pip -q install shap
#!pip install scikit-learn==1.1.3 -q

#загружаем библиотеки
import pandas as pd
#import shap
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
#import time
import os
#import phik

#from IPython.display import display, HTML
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, accuracy_score, confusion_matrix, recall_score, precision_score, precision_recall_curve
#from sklearn.metrics import roc_auc_score, make_scorer, RocCurveDisplay
#from statsmodels.stats.outliers_influence import variance_inflation_factor
#from sklearn.datasets import fetch_california_housing
#from phik.report import plot_correlation_matrix
#from typing import Optional

#from sklearn.model_selection import train_test_split, GridSearchCV
#from sklearn.pipeline import Pipeline
#from sklearn.compose import ColumnTransformer
#from sklearn.impute import SimpleImputer
#from sklearn.neighbors import KNeighborsClassifier
#from sklearn.ensemble import RandomForestClassifier
#from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
#from sklearn.svm import SVC

<div style="border:solid purple 5px; padding: 20px">
<h2 align="center"> Рубрика «Питонячий лайфхакер» <a class="tocSkip"> </h2>
<h3> Широкоформатный Jupyter <a class="tocSkip"> </h3>
    
Расширяем границы, или как сделать работу более комфортной (не всем нравится 😄)

    from IPython.core.display import display, HTML
    display(HTML("<style>.container { width:90% !important; }</style>"))


### Задаем константы

In [2]:
RANDOM_STATE=42

BUDGET = 10_000_000_000      # 10 млрд рублей
INCOME_PER_UNIT = 450_000    # 450 тыс. рублей за 1 тыс. баррелей
POINTS_TO_EXPLORE = 500
BEST_WELLS = 200

### Создаём функции

In [3]:
# Проверить общую информацию
def data_review(df, name):
    print(f'=== {name} ===')
    display(df.head(5))
    print()
    print(df.info())
    print()
    print('Пропуски:')
    print(df.isna().sum())
    print()
    print('Полные дубликаты:', df.duplicated().sum())
    print('Дубликаты id:', df['id'].duplicated().sum())
    print('-' * 50)

# Функция по обучению и предсказанию
#делит данные;
#обучает модель;
#делает предсказания;
#возвращает нужные метрики и данные.
def train_and_evaluate(df, region_name):
    features = df.drop(columns=['id', 'product'])
    target = df['product']
    
    X_train, X_valid, y_train, y_valid = train_test_split(
        features, target, test_size=0.25, random_state=RANDOM_STATE
    )
    
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    predictions = model.predict(X_valid)
    predictions = pd.Series(predictions, index=y_valid.index)
    
    rmse = mean_squared_error(y_valid, predictions, squared=False)
    mean_predicted = predictions.mean()
    
    print(f'--- {region_name} ---')
    print(f'Средний предсказанный запас: {mean_predicted:.2f}')
    print(f'RMSE модели: {rmse:.2f}')
    print()
    
    return y_valid.reset_index(drop=True), predictions.reset_index(drop=True), mean_predicted, rmse

# Функция по расчету прибыли
def calculate_profit(target, predictions, count=BEST_WELLS):
    predictions_sorted = predictions.sort_values(ascending=False)
    selected = target[predictions_sorted.index][:count]
    total_product = selected.sum()
    revenue = total_product * INCOME_PER_UNIT
    profit = revenue - BUDGET
    return profit

# Функция - Bootstrap: оценка прибыли и риска
def bootstrap_profit(target, predictions, n_samples=1000, sample_size=500):
    state = np.random.RandomState(42)
    profits_bootstrap = []

    target = pd.Series(target).reset_index(drop=True)
    predictions = pd.Series(predictions).reset_index(drop=True)

    for _ in range(n_samples):
        sample_indices = state.choice(target.index, size=sample_size, replace=True)

        target_sample = target[sample_indices].reset_index(drop=True)
        pred_sample = predictions[sample_indices].reset_index(drop=True)

        profit = calculate_profit(target_sample, pred_sample)
        profits_bootstrap.append(profit)

    profits_bootstrap = pd.Series(profits_bootstrap)

    mean_profit = profits_bootstrap.mean()
    lower_ci = profits_bootstrap.quantile(0.025)
    upper_ci = profits_bootstrap.quantile(0.975)
    risk_of_loss = (profits_bootstrap < 0).mean() * 100

    return profits_bootstrap, mean_profit, lower_ci, upper_ci, risk_of_loss

<div class="alert alert-success">
<h2> Комментарий ревьюера ✔️ <a class="tocSkip"> </h2>

Жирный плюс за автоматизацию 👏

## Загрузка и подготовка данных

### Загрузим данные

In [4]:
pth1 = '/datasets/geo_data_0.csv'
pth2 = '/datasets/geo_data_1.csv'
pth3 = '/datasets/geo_data_2.csv'

if os.path.exists(pth1):
    data_1 = pd.read_csv(pth1)
else:
    print('Файл geo_data_0.csv не найден')

if os.path.exists(pth2):
    data_2 = pd.read_csv(pth2)
else:
    print('Файл geo_data_1.csv не найден')

if os.path.exists(pth3):
    data_3 = pd.read_csv(pth3)
else:
    print('Файл geo_data_2.csv не найден')

In [5]:
data_review(data_1, 'Регион 1')
data_review(data_2, 'Регион 2')
data_review(data_3, 'Регион 3')

=== Регион 1 ===


,id,f0,f1,f2,product
0,txEyH,0.705745,-0.497823,1.221170,105.280062
1,2acmU,1.334711,-0.340164,4.365080,73.037750
2,409Wp,1.022732,0.151990,1.419926,85.265647
3,iJLyR,-0.032172,0.139033,2.978566,168.620776
4,Xdl7t,1.988431,0.155413,4.751769,154.036647



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None

Пропуски:
id         0
f0         0
f1         0
f2         0
product    0
dtype: int64

Полные дубликаты: 0
Дубликаты id: 10
--------------------------------------------------
=== Регион 2 ===


,id,f0,f1,f2,product
0,kBEdx,-15.001348,-8.276000,-0.005876,3.179103
1,62mP7,14.272088,-3.475083,0.999183,26.953261
2,vyE1P,6.263187,-5.948386,5.001160,134.766305
3,KcrkZ,-13.081196,-11.506057,4.999415,137.945408
4,AHL4O,12.702195,-8.147433,5.004363,134.766305



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None

Пропуски:
id         0
f0         0
f1         0
f2         0
product    0
dtype: int64

Полные дубликаты: 0
Дубликаты id: 4
--------------------------------------------------
=== Регион 3 ===


,id,f0,f1,f2,product
0,fwXo0,-1.146987,0.963328,-0.828965,27.758673
1,WJtFt,0.262778,0.269839,-2.530187,56.069697
2,ovLUW,0.194587,0.289035,-5.586433,62.871910
3,q6cA6,2.236060,-0.553760,0.930038,114.572842
4,WPMUX,-0.515993,1.716266,5.899011,149.600746



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None

Пропуски:
id         0
f0         0
f1         0
f2         0
product    0
dtype: int64

Полные дубликаты: 0
Дубликаты id: 4
--------------------------------------------------


<div class="alert alert-success">
<h2> Комментарий ревьюера ✔️ <a class="tocSkip"> </h2>

На пропуски можно было посмотреть визуально. Для этого я использовал бы библиотеку [seaborn](https://seaborn.pydata.org/), а сам код выглядел бы как-то так
    
```python
import seaborn as sns
sns.heatmap(banks_data.isna(), yticklabels=False, cbar=False, cmap="YlGnBu");
```

**📋 Выводы по загрузке данных**

Предварительный обзор данных показал, что во всех трёх регионах содержится по 100 000 наблюдений и по 5 признаков: `id`, `f0`, `f1`, `f2` и целевой признак `product`.

Пропуски во всех столбцах отсутствуют, типы данных корректны: числовые признаки и целевой показатель имеют тип `float64`, идентификатор скважины `id` — строковый тип `object`. Явных проблем со структурой данных не обнаружено.

Полных дубликатов строк в датасетах нет. Это говорит о том, что полностью повторяющиеся наблюдения отсутствуют. Однако необходима отдельная проверка на дубликаты по полю `id`.

В целом данные выглядят чистыми и пригодными для построения модели. Для обучения можно использовать признаки `f0`, `f1`, `f2`, а столбец `id` следует исключить, так как он не несёт полезной информации для предсказания запасов сырья.

### Анализ дупликатов

In [6]:
for i, df in enumerate([data_1, data_2, data_3], start=1):
    print(f'Регион {i} — дубликаты id: {df["id"].duplicated().sum()}')

Регион 1 — дубликаты id: 10
Регион 2 — дубликаты id: 4
Регион 3 — дубликаты id: 4


In [7]:
for i, df in enumerate([data_1, data_2, data_3]):
    duplicated_wells = df[df['id'].duplicated(keep=False)].sort_values('id')
    
    print(f'===== Регион {i} — скважины с дублирующимися id =====')
    display(duplicated_wells)
    print(f'Количество строк: {len(duplicated_wells)}')
    print(f'Количество уникальных дублирующихся id: {duplicated_wells["id"].nunique()}')
    print()

===== Регион 0 — скважины с дублирующимися id =====


,id,f0,f1,f2,product
66136,74z30,1.084962,-0.312358,6.990771,127.643327
64022,74z30,0.741456,0.459229,5.153109,140.771492
51970,A5aEY,-0.180335,0.935548,-2.094773,33.020205
3389,A5aEY,-0.039949,0.156872,0.209861,89.249364
69163,AGS9W,-0.933795,0.116194,-3.655896,19.230453
42529,AGS9W,1.454747,-0.479651,0.683380,126.370504
931,HZww2,0.755284,0.368511,1.863211,30.681774
7530,HZww2,1.061194,-0.373969,10.430210,158.828695
63593,QcMuo,0.635635,-0.473422,0.862670,64.578675
1949,QcMuo,0.506563,-0.323775,-2.215583,75.496502


Количество строк: 20
Количество уникальных дублирующихся id: 10

===== Регион 1 — скважины с дублирующимися id =====


,id,f0,f1,f2,product
5849,5ltQ6,-3.435401,-12.296043,1.999796,57.085625
84461,5ltQ6,18.213839,2.191999,3.993869,107.813044
1305,LHZR0,11.170835,-1.945066,3.002872,80.859783
41906,LHZR0,-8.989672,-4.286607,2.009139,57.085625
2721,bfPNe,-9.494442,-5.463692,4.006042,110.992147
82178,bfPNe,-6.202799,-4.820045,2.995107,84.038886
47591,wt4Uk,-9.091098,-8.109279,-0.002314,3.179103
82873,wt4Uk,10.259972,-9.376355,4.994297,134.766305


Количество строк: 8
Количество уникальных дублирующихся id: 4

===== Регион 2 — скважины с дублирующимися id =====


,id,f0,f1,f2,product
45404,KUPhW,0.231846,-1.698941,4.990775,11.716299
55967,KUPhW,1.211150,3.176408,5.543540,132.831802
11449,VF7Jo,2.122656,-0.858275,5.746001,181.716817
49564,VF7Jo,-0.883115,0.560537,0.723601,136.233420
44378,Vcm5J,-1.229484,-2.439204,1.222909,137.968290
95090,Vcm5J,2.587702,1.986875,2.482245,92.327572
28039,xCHr8,1.633027,0.368135,-2.378367,6.120525
43233,xCHr8,-0.847066,2.101796,5.597130,184.388641


Количество строк: 8
Количество уникальных дублирующихся id: 4



**Анализ по дупликатам уникальных id скважин**

Дополнительно была выполнена проверка уникальности идентификаторов скважин `id`. Во всех трёх регионах есть повторяющиеся идентификаторы:
- в регионе 1 — 10 дублирующихся `id`;
- в регионе 2 — 4 дублирующихся `id`;
- в регионе 3 — 4 дублирующихся `id`.

При этом строки с одинаковыми `id` не являются полными дубликатами: значения признаков `f0`, `f1`, `f2` и целевого признака `product` у них различаются. Это означает, что одинаковый идентификатор встречается у разных наблюдений, поэтому считать такие записи полными копиями и удалять их без дополнительного обоснования нельзя.

Так как столбец `id` не участвует в обучении модели и не несёт прогностической ценности, наличие повторяющихся идентификаторов не является критичной проблемой для построения линейной регрессии. В дальнейшем для обучения модели будут использоваться только признаки `f0`, `f1`, `f2`, а столбец `id` будет исключён.

<div class="alert alert-success">
<h2> Комментарий ревьюера ✔️ <a class="tocSkip"> </h2>

Логичное решение 👍🏼

### 📋 Выводы этапа «Загрузки данных»
В целом данные хорошо подготовлены для моделирования: пропуски отсутствуют, типы данных корректны, полные дубликаты не обнаружены. Единственная особенность — наличие повторяющихся значений `id`, однако, поскольку этот признак не используется при обучении модели, это не помешает дальнейшему анализу.

## Обучение и проверка модели

### Обучим модели по каждому региону

In [8]:
targets = []
predictions = []
mean_preds = []
rmses = []

for i, df in enumerate([data_1, data_2, data_3], start=1):
    target, pred, mean_pred, rmse = train_and_evaluate(df, f'Регион {i}')
    
    targets.append(target)
    predictions.append(pred)
    mean_preds.append(mean_pred)
    rmses.append(rmse)
    
results = pd.DataFrame({
    'region': [f'Регион {i}' for i in range(1, len(mean_preds) + 1)],
    'mean_predicted_product': mean_preds,
    'rmse': rmses
})

display(results)

--- Регион 1 ---
Средний предсказанный запас: 92.40
RMSE модели: 37.76

--- Регион 2 ---
Средний предсказанный запас: 68.71
RMSE модели: 0.89

--- Регион 3 ---
Средний предсказанный запас: 94.77
RMSE модели: 40.15



,region,mean_predicted_product,rmse
0,Регион 1,92.398800,37.756600
1,Регион 2,68.712878,0.890280
2,Регион 3,94.771024,40.145872


<div class="alert alert-success">
<h2> Комментарий ревьюера ✔️ <a class="tocSkip"> </h2>

Круто, что ты фиксируешь `random_state`. Кстати, его можно задать один раз в начале проекта, например, используя любимый нами [numpy](https://stackoverflow.com/questions/21494489/what-does-numpy-random-seed0-do)

### 📋 Выводы этапа «Загрузки данных»
Для каждого региона была обучена модель линейной регрессии, после чего получены предсказания на валидационной выборке. Далее были рассчитаны средний предсказанный запас сырья и значение RMSE.

Результаты показали, что:
- для региона 1 средний предсказанный запас составил 92.40 тыс. баррелей, RMSE — 37.76;
- для региона 2 средний предсказанный запас составил 68.71 тыс. баррелей, RMSE — 0.89;
- для региона 3 средний предсказанный запас составил 94.77 тыс. баррелей, RMSE — 40.15.

Наименьшая ошибка прогноза наблюдается у модели для региона 2. Это означает, что именно в этом регионе линейная регрессия предсказывает объём запасов наиболее точно. При этом средний предсказанный запас в регионе 2 заметно ниже, чем в регионах 1 и 3.

В регионах 1 и 3 средний предсказанный запас выше, однако и ошибка модели там существенно больше. Следовательно, на данном этапе нельзя сделать вывод о лучшем регионе только по среднему запасу: необходимо дополнительно оценить потенциальную прибыль и риск убытков.

## Подготовка к расчёту прибыли

### Достаточный объём сырья для безубыточности

In [9]:
#Найдём, сколько в среднем должна давать одна скважина, чтобы разработка 200 скважин окупилась.
break_even_product = BUDGET / (BEST_WELLS * INCOME_PER_UNIT)
print(f'Безубыточный объём на одну скважину: {break_even_product:.2f} тыс. баррелей')

Безубыточный объём на одну скважину: 111.11 тыс. баррелей


In [10]:
#Сравним со средним фактическим запасом по регионам:
mean_products = []

for i, df in enumerate([data_1, data_2, data_3], start=1):
    mean_product = df['product'].mean()
    mean_products.append(mean_product)
    print(f'Регион {i} — средний запас: {mean_product:.2f} тыс. баррелей')

mean_products_table = pd.DataFrame({
    'region': [f'Регион {i}' for i in range(1, 4)],
    'mean_product': mean_products
})

display(mean_products_table)

Регион 1 — средний запас: 92.50 тыс. баррелей
Регион 2 — средний запас: 68.83 тыс. баррелей
Регион 3 — средний запас: 95.00 тыс. баррелей


,region,mean_product
0,Регион 1,92.500
1,Регион 2,68.825
2,Регион 3,95.000


### 📋 Выводы этапа «Подготовка к расчёту прибыли»

Для безубыточной разработки одной скважины необходим объём сырья 111.11 тыс. баррелей.

Средний фактический запас во всех трёх регионах ниже этого значения:
- в регионе 1 — 92.50 тыс. баррелей;
- в регионе 2 — 68.83 тыс. баррелей;
- в регионе 3 — 95.00 тыс. баррелей.

Это означает, что при случайном выборе скважин разработка региона не будет безубыточной. Следовательно, для окупаемости проекта необходимо отбирать только лучшие скважины на основе прогнозов модели.

<div class="alert alert-success">
<h2> Комментарий ревьюера ✔️ <a class="tocSkip"> </h2>

Так и есть!

## Расчёт прибыли и рисков 

### Расчитаем прибыль для каждого региона

По условию:
* исследуют 500 точек;
* выбирают 200 лучших по предсказаниям;
* прибыль считаем уже по реальным запасам этих скважин.

In [11]:
profits = []

for i, (target, pred) in enumerate(zip(targets, predictions), start=1):
    #по умолчанию в функции заложена константа count=BEST_WELLS = 200 по условиям задачи
    profit = calculate_profit(target, pred)
    profits.append(profit)

print('Прибыль по регионам:')
for i, profit in enumerate(profits, start=1):
    print(f'Регион {i}: {profit:,.0f} руб.')
    
profits_table = pd.DataFrame({
    'region': [f'Регион {i}' for i in range(1, len(profits) + 1)],
    'profit': profits
})


display(
    profits_table.style.format({
        'profit': lambda x: f'{x:,.0f} руб.'.replace(',', ' ')
    })
)

Прибыль по регионам:
Регион 1: 3,359,141,114 руб.
Регион 2: 2,415,086,697 руб.
Регион 3: 2,598,571,759 руб.


,region,profit
0,Регион 1,3 359 141 114 руб.
1,Регион 2,2 415 086 697 руб.
2,Регион 3,2 598 571 759 руб.


<div style="border:solid purple 5px; padding: 20px">
<h2 align="center"> Рубрика «Питонячий лайфхакер» <a class="tocSkip"> </h2>
    
<h3> Библиотека os <a class="tocSkip"> </h3>

Модуль os в Python — это библиотека функций для работы с операционной системой. Методы, включенные в неё позволяют определять тип операционной системы, получать доступ к переменным окружения, управлять директориями и файлами:

- проверка существования объекта по заданному пути;
- определение размера в байтах;
- удаление;
- переименование и др.

Библиотека довольно обширная, при желании, можно почитать [официальную документацию](https://docs.python.org/3/library/os.html), либо короткий [перевод](https://pythonworld.ru/moduli/modul-os.html). Вот некоторые методы этой библиотеки

Чтобы узнать, какая она на вашей операционной системе, используется функция `getcwd`

![](https://i.ibb.co/4RYXGqV/image.png)

Функция `chdir` сменит ее на другую, которая будет указана аргументом

![](https://i.ibb.co/rwM0WY8/image.png)

Функция `listdir` без указания аргумента покажет все файлы и папки текущей рабочей директории. Можно также указать аргументом интересующий вас путь

![](https://i.ibb.co/4pWQ16Y/image.png)

Чтобы создать папку, применяется функция `mkdir`. Эта функция сможет создать только одну папку в существующем пути, если указанного пути не существует, то вызовется ошибка. Если папка уже существует - вызовется ошибка

![](https://i.ibb.co/q53Z6YD/image.png)

### Применить технику Bootstrap для расчета рисков и прибыли для каждого региона

In [12]:
bootstrap_distributions = []
mean_profits = []
lower_cis = []
upper_cis = []
risks = []

for i, (target, pred) in enumerate(zip(targets, predictions), start=1):
    profits_bootstrap, mean_profit, lower_ci, upper_ci, risk_of_loss = bootstrap_profit(target, pred)

    bootstrap_distributions.append(profits_bootstrap)
    mean_profits.append(mean_profit)
    lower_cis.append(lower_ci)
    upper_cis.append(upper_ci)
    risks.append(risk_of_loss)

    print(f'--- Регион {i} ---')
    print(f'Средняя прибыль: {mean_profit:,.0f} руб.'.replace(',', ' '))
    print(f'95%-й доверительный интервал: ({lower_ci:,.0f}, {upper_ci:,.0f}) руб.'.replace(',', ' '))
    print(f'Риск убытков: {risk_of_loss:.2f}%')
    print()

--- Регион 1 ---
Средняя прибыль: 399 575 478 руб.
95%-й доверительный интервал: (-110 467 895  897 460 328) руб.
Риск убытков: 6.00%

--- Регион 2 ---
Средняя прибыль: 452 048 891 руб.
95%-й доверительный интервал: (61 684 480  845 340 178) руб.
Риск убытков: 1.50%

--- Регион 3 ---
Средняя прибыль: 375 009 903 руб.
95%-й доверительный интервал: (-144 766 727  888 390 404) руб.
Риск убытков: 8.00%



<div class="alert alert-success">
<h2> Комментарий ревьюера ✔️ <a class="tocSkip"> </h2>

Я бы еще добавил гистограммы распределений, на которые можно вынести вертикальные линии для средних и доверительного интервала. Я бы использовал для этого [axvline](https://stackoverflow.com/questions/24988448/how-to-draw-vertical-lines-on-a-given-plot)

In [13]:
bootstrap_results = pd.DataFrame({
    'region': [f'Регион {i}' for i in range(1, len(mean_profits) + 1)],
    'mean_profit': mean_profits,
    'lower_95_ci': lower_cis,
    'upper_95_ci': upper_cis,
    'risk_of_loss_%': risks
})

display(
    bootstrap_results.style.format({
        'mean_profit': lambda x: f'{x:,.0f} руб.'.replace(',', ' '),
        'lower_95_ci': lambda x: f'{x:,.0f} руб.'.replace(',', ' '),
        'upper_95_ci': lambda x: f'{x:,.0f} руб.'.replace(',', ' '),
        'risk_of_loss_%': '{:.2f}%'
    })
)

,region,mean_profit,lower_95_ci,upper_95_ci,risk_of_loss_%
0,Регион 1,399 575 478 руб.,-110 467 895 руб.,897 460 328 руб.,6.00%
1,Регион 2,452 048 891 руб.,61 684 480 руб.,845 340 178 руб.,1.50%
2,Регион 3,375 009 903 руб.,-144 766 727 руб.,888 390 404 руб.,8.00%


### Выбор подходящего региона

Оставляем только регионы, где риск убытков меньше 2.5%.

In [14]:
best_regions = bootstrap_results[bootstrap_results['risk_of_loss_%'] < 2.5]
display(
    best_regions.style.format({
        'mean_profit': lambda x: f'{x:,.0f} руб.'.replace(',', ' '),
        'lower_95_ci': lambda x: f'{x:,.0f} руб.'.replace(',', ' '),
        'upper_95_ci': lambda x: f'{x:,.0f} руб.'.replace(',', ' '),
        'risk_of_loss_%': '{:.2f}%'
    })
)

,region,mean_profit,lower_95_ci,upper_95_ci,risk_of_loss_%
1,Регион 2,452 048 891 руб.,61 684 480 руб.,845 340 178 руб.,1.50%


Если таких регионов несколько, выбираем тот, где средняя прибыль максимальна.

In [15]:
best_region = best_regions.sort_values(by='mean_profit', ascending=False).iloc[0]
print(
    f'Рекомендуемый регион для разработки: {best_region["region"]}\n'
    f'Средняя прибыль: {best_region["mean_profit"]:,.0f} руб.\n'
    f'Риск убытков: {best_region["risk_of_loss_%"]:.2f}%'
    .replace(',', ' ')
)

Рекомендуемый регион для разработки: Регион 2
Средняя прибыль: 452 048 891 руб.
Риск убытков: 1.50%


### 📋 Выводы этапа «Расчёт прибыли и рисков» 

С помощью Bootstrap для каждого региона были рассчитаны средняя прибыль, 95%-й доверительный интервал и риск убытков.

Результаты показали:
- **Регион 1**: средняя прибыль составляет 399 575 478 руб., 95%-й доверительный интервал — от -110 467 895 до 897 460 328 руб., риск убытков — 6.00%;
- **Регион 2**: средняя прибыль составляет 452 048 891 руб., 95%-й доверительный интервал — от 61 684 480 до 845 340 178 руб., риск убытков — 1.50%;
- **Регион 3**: средняя прибыль составляет 375 009 903 руб., 95%-й доверительный интервал — от -144 766 727 до 888 390 404 руб., риск убытков — 8.00%.

По условию задачи для разработки подходят только те регионы, где вероятность убытков меньше 2.5%. Этому критерию соответствует только **Регион 2**.

Кроме того, у региона 2 не только допустимый риск убытков, но и наибольшая средняя прибыль среди регионов, удовлетворяющих условию. Следовательно, **наиболее перспективным регионом для бурения новых скважин является Регион 2**.

<div class="alert alert-success">
<h2> Комментарий ревьюера ✔️ <a class="tocSkip"> </h2>


![](https://i.gifer.com/7V3.gif)


## Чек-лист готовности проекта

Поставьте 'x' в выполненных пунктах. Далее нажмите Shift+Enter.

- [x]  Jupyter Notebook открыт
- [x]  Весь код выполняется без ошибок
- [x]  Ячейки с кодом расположены в порядке исполнения
- [x]  Выполнен шаг 1: данные подготовлены
- [x]  Выполнен шаг 2: модели обучены и проверены
    - [x]  Данные корректно разбиты на обучающую и валидационную выборки
    - [x]  Модели обучены, предсказания сделаны
    - [x]  Предсказания и правильные ответы на валидационной выборке сохранены
    - [x]  На экране напечатаны результаты
    - [x]  Сделаны выводы
- [x]  Выполнен шаг 3: проведена подготовка к расчёту прибыли
    - [x]  Для всех ключевых значений созданы константы Python
    - [x]  Посчитано минимальное среднее количество продукта в месторождениях региона, достаточное для разработки
    - [x]  По предыдущему пункту сделаны выводы
    - [x]  Написана функция расчёта прибыли
- [x]  Выполнен шаг 4: посчитаны риски и прибыль
    - [x]  Проведена процедура *Bootstrap*
    - [x]  Все параметры бутстрепа соответствуют условию
    - [x]  Найдены все нужные величины
    - [x]  Предложен регион для разработки месторождения
    - [x]  Выбор региона обоснован